![Cabecera](../../assets/cabecera_sql.png)

# SQL con Python: Ordenación, agregación y agrupación 

Vamos con una sesión con varios aspectos interesantes: ordenar la salida y agregar valores.  
 Antes carguemos nuestros datos, librerías y la función tan util para ejecutar queries:

In [1]:
import pandas as pd
import sqlite3

# Conectamos con la base de datos chinook.db
connection = sqlite3.connect("data/chinook.db")

# Obtenemos un cursor que utilizaremos para hacer las queries
cursor_bootcamp = connection.cursor()

In [2]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    cursor_bootcamp.execute(query) # Recuerda que sólo funcionará si has llamado cursor_bootcamp
                                    # a tu cursor, si no, cambia el nombre en todo el código de la función

    # Almacena los datos de la query 
    ans = cursor_bootcamp.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in cursor_bootcamp.description]

    return pd.DataFrame(ans,columns=names)

### ORDER BY
Podemos **ordenar la tabla por el campo/s que queramos**. Por defecto ordena alfabéticamente los strings y de menor a mayor los tipos numéricos. Si quieres que ordene al revés, tienes que poner `DESC` de la forma `ORDER BY campo DESC` (como el `ascending = False` del `sort_values` de pandas,o el `reverse = True` en el caso de `sort` de listas)

In [3]:
# Obtengamos los datos de la tabla "tracks" ordenados por nombre de forma descendente
query = '''
SELECT *
FROM tracks
ORDER BY name DESC
'''
sql_query(query)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1077,Último Pau-De-Arara,85,1,10,Corumbá/José Gumarães/Venancio,200437,6638563,0.99
1,1073,Óia Eu Aqui De Novo,85,1,10,None,219454,7469735,0.99
2,2078,Óculos,169,1,7,None,219271,7262419,0.99
3,3496,"Étude 1, In C Major - Preludio (Presto) - Liszt",340,4,24,None,51780,2229617,0.99
4,333,É que Nessa Encarnação Eu Nasci Manga,29,1,9,Lucina/Luli,196519,6568081,0.99
...,...,...,...,...,...,...,...,...,...
3498,3254,#9 Dream,255,2,9,None,278312,4506425,0.99
3499,109,#1 Zero,11,1,4,"Cornell, Commerford, Morello, Wilk",299102,9731988,0.99
3500,3412,"""Eine Kleine Nachtmusik"" Serenade In G, K. 525...",281,2,24,Wolfgang Amadeus Mozart,348971,5760129,0.99
3501,2918,"""?""",231,3,19,None,2782333,528227089,1.99


In [4]:
# Obtengamos el nombre de los clientes norteamericanos ordenados por apellido
query = '''
SELECT FirstName "Nombre", LastName "Apellido", Country "País"
FROM Customers
WHERE Country in ("USA", "Canada")
ORDER BY LastName
'''
sql_query(query)

,Nombre,Apellido,País
0,Julia,Barnett,USA
1,Michelle,Brooks,USA
2,Robert,Brown,Canada
3,Kathy,Chase,USA
4,Richard,Cunningham,USA
5,Edward,Francis,Canada
6,John,Gordon,USA
7,Tim,Goyer,USA
8,Patrick,Gray,USA
9,Frank,Harris,USA


### Agregaciones
Como ocurre cuando trabajamos con datos en ocasiones nos interesa obtener algún estadístico como el máximo de un campo, su desviación estándar o simplemente un conteo de registros no nulos. Para ello podemos usar funciones como `MAX`, `COUNT` o `AVG`. En [esta página](https://www.sqlservertutorial.net/sql-server-aggregate-functions/) encontrarás un resumen con las principales funciones.

In [5]:
# Obtener el número de canciones que contienen la palabra love
query = '''
SELECT COUNT(*)
FROM tracks
WHERE name LIKE "%love%"
'''
sql_query(query)

,COUNT(*)
0,114


In [6]:
# Encontrar la media del precio de las canciones compradas
query = '''
SELECT AVG(unitprice)
FROM invoice_items
'''
sql_query(query)

,AVG(unitprice)
0,1.039554


In [7]:
# Obtener la duración máxima de las canciones
query = '''
SELECT MAX(Milliseconds)/1000
FROM tracks
'''
sql_query(query)

,MAX(Milliseconds)/1000
0,5286


### Agrupaciones (GROUP BY):
Para terminar veamos como se hacen agrupaciones empleando GROUP BY, y de forma muy similar a como se hace con pandas.


In [8]:
# Selección del precio unitario en funcion del género
query = '''
SELECT GenreId, SUM(unitprice) as TOT_PRICE
FROM tracks
GROUP BY GenreId
ORDER BY TOT_PRICE DESC
LIMIT 10
'''
sql_query(query)

,GenreId,TOT_PRICE
0,1,1284.03
1,7,573.21
2,3,370.26
3,4,328.68
4,19,185.07
5,2,128.70
6,21,127.36
7,6,80.19
8,24,73.26
9,14,60.39


O calcular cuantas canciones hay por compositor

In [9]:
query = '''
SELECT composer, COUNT(trackid)
FROM tracks
WHERE Composer IS NOT NULL
GROUP BY composer
ORDER BY 2 DESC
LIMIT 20
'''
sql_query(query)

,Composer,COUNT(trackid)
0,Steve Harris,80
1,U2,44
2,Jagger/Richards,35
3,Billy Corgan,31
4,Kurt Cobain,26
5,Bill Berry-Peter Buck-Mike Mills-Michael Stipe,25
6,The Tea Party,24
7,Miles Davis,23
8,Gilberto Gil,23
9,Chris Cornell,23


Podrás practicar con todo lo visto en los ejercicios correspondientes, en los que además practicarás con la combinación de queries y subqueries.